# قسمت دوم — آزمایش تاریخچه بدون اینترنت

**تمام داده‌ها مصنوعی‌اند.** تاریخ‌ها، قیمت‌ها و روزهای موجود در نمونه، شاهد معاملات واقعی نیستند. این دفترچه نصب، قرارداد داده، بازه تاریخی، ذخیره و آفلاین را بررسی می‌کند.

## سلول ۱ — ورود کتابخانه‌ها

In [ ]:
from datetime import timedelta
from tempfile import TemporaryDirectory
from unittest.mock import patch
import httpx
import pandas as pd
import pydara
from pydara.tse import FetchPolicy, FreshnessPolicy
from pydara.tse.samples import sample_client

print("نسخه:", pydara.__version__)
assert pydara.__version__ == "0.7.3a6", "Install this bundle and restart the kernel."

## سلول ۲ — تابع نمایش

In [ ]:
def show_history(frame):
    columns = ["date", "last_price", "closing_price", "volume", "trade_count"]
    if "jdate" in frame.columns:
        columns.insert(1, "jdate")
    print(f"تعداد ردیف‌ها: {len(frame):,}")
    display(frame[columns])
    print("وضعیت پوشش:", frame.attrs["pydara"]["coverage_status"])

## سلول ۳ — مقادیر ثابت

In [ ]:
SYMBOL = "فولاد"
SYMBOLS = ["فولاد", "نمادناموجود", "شستا", "فولاد"]
START_JALALI = "1403-01-01"
END_JALALI = "1403-12-30"

## سلول ۴ — اجرای دریافت مصنوعی

In [ ]:
with sample_client() as client:
    df = client.get_history(SYMBOL, include_jdate=True)
    selected = client.get_history(
        SYMBOL, start=START_JALALI, end=END_JALALI,
        calendar="jalali", include_jdate=True,
    )
    batch = client.get_history_many(SYMBOLS, on_error="collect")
    chunks = list(client.iter_history(
        SYMBOL, start="2024-03-19", end="2025-03-21", chunk_days=200,
    ))

assert len(df) == 6
assert len(selected) == 4
assert batch.requests["status"].tolist() == ["ok", "error", "ok", "ok"]
assert len(batch.data) == 18

## سلول ۵ — نمایش نتیجه

In [ ]:
show_history(df)
show_history(selected)
display(batch.requests)
display(batch.errors)
print("تعداد ردیف‌های هر قطعه:", [len(chunk) for chunk in chunks])
print("هشدارهای اولین ردیف:", df.loc[0, "quality_flags"])

## سلول ۶ — ذخیره نمونه در پوشه موقت

پایگاه واقعی قسمت اول تغییر نمی‌کند. `flush` برای تأیید پایان نوشتن است.

In [ ]:
temporary_store = TemporaryDirectory(prefix="pydara_history_demo_")
with sample_client(storage=True, data_dir=temporary_store.name) as client:
    stored = client.get_history(SYMBOL)
    report = client.flush()
    print("ثبت تکمیل شد:", report.complete)
    print(client.storage_status())
assert report.complete

## سلول ۷ — بازیابی آفلاین با مسدودکردن مسیر شبکه

In [ ]:
def block_network(*args, **kwargs):
    raise AssertionError("Unexpected network request")

offline_policy = FetchPolicy(
    mode="offline",
    freshness=FreshnessPolicy(max_age=timedelta(0), allow_stale=True),
)
with patch.object(httpx.Client, "send", new=block_network):
    with sample_client(storage=True, data_dir=temporary_store.name) as client:
        cached = client.get_history(SYMBOL, policy=offline_policy)
        assert client.request_count == 0
        assert cached["from_cache"].all()
        assert cached["is_stale"].all()

assert stored[["date", "last_price", "closing_price"]].equals(
    cached[["date", "last_price", "closing_price"]]
)
show_history(cached)
print("هیچ درخواست شبکه‌ای ارسال نشد.")
temporary_store.cleanup()

هشدار `StaleDataWarning` در سلول قبل عمدی است: داده کش با اجازه صریح بازیابی شده و به‌روز فرض نشده است. قدم بعد: دفترچه `04_history_live.ipynb`.